# 🧠 Epoch Analysis & Generalization Curves

Welcome to the hands-on explanation notebook for **Epoch Analysis and Generalization Curves**! In this notebook, we will:
1. Clarify the definitions and mathematical relationship between epochs, batch size, and update steps.
2. Define a mathematical function representing overfitting, modeling a monotonic training decay vs. a U-shaped validation curve.
3. Write an **Early Stopping Detector from scratch** in Python, mimicking the `patience` mechanism in deep learning libraries.
4. Visualize the **Overfitting Zone** and the **Early Stopping Trigger** on a clear dual-line loss plot.
5. Connect these concepts to YOLO's default `epochs=100` and `patience=100` training parameters.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Simulating Overfitting Loss Metrics

We model the losses mathematically over 80 epochs:
-   **Training Loss:** Decays exponentially as the model fits the training set.
-   **Validation Loss:** Decays early, reaches a minimum, and then starts to rise as the model memorizes training noise (overfitting).

In [ ]:
epochs = np.arange(1, 81)

# Training loss decreases monotonically
train_loss = 2.0 * np.exp(-0.08 * epochs) + 0.05 + np.random.normal(0, 0.005, len(epochs))

# Validation loss reaches a minimum around epoch 30, then rises
val_loss = 2.1 * np.exp(-0.08 * epochs) + 0.08 * np.exp(0.025 * epochs) + np.random.normal(0, 0.01, len(epochs))

## 2. Implementing Early Stopping with Patience

Early stopping monitors validation loss. If validation loss does not improve for a sequence of `patience` epochs, training is halted, and the weights of the best epoch are restored.

In [ ]:
def find_early_stopping(val_losses, patience=10):
    best_loss = float('inf')
    best_epoch = 0
    no_improvement_count = 0
    
    for idx, loss in enumerate(val_losses):
        epoch = idx + 1
        if loss < best_loss:
            best_loss = loss
            best_epoch = epoch
            no_improvement_count = 0
        else:
            no_improvement_count += 1
            
        if no_improvement_count >= patience:
            return best_epoch, epoch
            
    return best_epoch, len(val_losses)

# Calculate stopping metrics for patience = 8
best_ep, stop_ep = find_early_stopping(val_loss, patience=8)
print(f"Optimal Epoch (Minimum Val Loss): {best_ep}")
print(f"Early Stopping Triggered at Epoch: {stop_ep}")

## 3. Visualizing Generalization Curves and the Overfitting Zone

Let's plot the curves, shading the **Overfitting Zone** (where validation loss starts climbing) and marking the early stopping point.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_loss, color='blue', linewidth=2, label='Training Loss')
plt.plot(epochs, val_loss, color='red', linewidth=2, label='Validation Loss')

plt.axvline(x=best_ep, color='green', linestyle='--', linewidth=2, label=f'Optimal Model (Epoch {best_ep})')
plt.axvline(x=stop_ep, color='purple', linestyle=':', linewidth=2, label=f'Stopping Trigger (Epoch {stop_ep})')

plt.axvspan(best_ep, len(epochs), color='red', alpha=0.1, label='Overfitting Zone')

plt.xlabel('Epochs')
plt.ylabel('Loss Value')
plt.title('Epoch Analysis: Overfitting Detection & Early Stopping')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

Look at the visualization:
-   **Before Epoch 30 (Optimal Model):** Both training and validation losses decrease. The model is learning generalized features.
-   **After Epoch 30 (Overfitting Zone):** Training loss continues falling, but validation loss rises. The model is memorizing training-set details.
-   **Epoch 38 (Stopping Trigger):** At this point, the validation loss has failed to make a new minimum for 8 consecutive epochs, triggering the stopping condition to save compute and prevent further overfitting.

## 💡 Connection to YOLO and Deep Learning
*   **YOLO Early Stopping:** Inside the YOLO trainer, the default settings configure `patience=100`. If you set `epochs=300` but the validation mAP (or loss) stops improving for 100 epochs, YOLO will automatically cease training, outputting the best weights saved at the peak validation epoch.
*   This prevents wasting hours of GPU execution time on a model that is already degrading in generalization ability.